In [45]:
import pandas as pd
import numpy as np
import pint
ureg = pint.UnitRegistry()

In [46]:
d1 = pd.read_csv("data/forcedecks_anomaly_data_full.csv")
d2 = pd.read_csv("data/forcedecks_anomaly_data_simplified.csv")
d3 = pd.read_csv("data/Forceplate_Invalid tests.csv")
d4 = pd.read_csv("data/Forceplate_Invalid_Tests_Olympic.csv")

In [47]:
#renaming columns for readability

rename_map = {
    "Unnamed: 0":                             "row_id",
    "Name":                                   "name",
    "Date of InValid Test":                   "date",
    "Sport":                                  "sport",
    "ExternalId":                             "external_id",
    "Test.Type":                              "test_type",
    "Date":                                   "date",
    "Time":                                   "time",
    "BW..KG.":                                "bw_kg",
    "Reps":                                   "reps",
    "Tags":                                   "tags",
    "Additional.Load..lb.":                   "additional_load_lb",
    "Jump.Height..Imp.Dis...cm..":            "jump_height_cm",
    "Peak.Power..W..":                        "peak_power_w",
    "Peak.Power...BM..W.kg..":               "peak_power_per_bm",
    "RSI.modified..m.s..":                    "rsi_modified",
    "Concentric.Impulse..N.s..":              "concentric_impulse",
    "Flight.Time.Contraction.Time.":          "flight_contraction_ratio",
    "Force.at.Zero.Velocity..N..":            "force_at_zero_vel",
    "Eccentric.Duration..ms..":               "eccentric_duration_ms",
    "Eccentric.Peak.Power...BM..W.kg..":     "ecc_peak_power_per_bm",
    "Eccentric.Deceleration.RFD....Asym.....":"ecc_dec_rfd_asym",
    "Concentric.Impulse....Asym.....":        "conc_impulse_asym",
    "Concentric.RFD....Asym.....":            "conc_rfd_asym",
    "Force.at.Zero.Velocity....Asym.....":    "force_zero_vel_asym",
    "Peak.Landing.Force....Asym.....":        "peak_landing_force_asym",
    "Eccentric.Braking.RFD....Asym.....":    "ecc_braking_rfd_asym",
}

In [48]:
d1 = d1.rename(columns=rename_map)
d2 = d2.rename(columns=rename_map)
d3 = d3.rename(columns=rename_map)
d4 = d4.rename(columns=rename_map)


#cleaning names
d1['name'] = d1['name'].str.strip().str.lower()
d2['name'] = d2['name'].str.strip().str.lower()
d3['name'] = d3['name'].str.strip().str.lower()
d4['name'] = d4['name'].str.strip().str.lower()



In [49]:
#Cleaning full data set

d1['date'] = pd.to_datetime(d1['date'])
#d1['time'] = pd.to_datetime(d1['time'], format='mixed')
d1['datetime'] = pd.to_datetime(
    d1['date'].astype(str) + ' ' + d1['time'].astype(str),
    format="%Y-%m-%d %I:%M %p"
)

d1['name'] = d1['name'].str.lower()
d1['name'] = d1['name'].str.rstrip()
d1['name'] = d1['name'].str.replace(r'\s+', ' ', regex=True).str.strip()
d1.loc[d1["name"] == "fiona o'keeffe", 'name'] = "fiona o'keefe"
d1.loc[d1['name'] == 'christopher miller', 'name'] = 'chris miller'
d1.loc[d1['name'] == 'steve simpkins jr.', 'name'] = 'steve simpkins'
#d1["bw_lg"] = d1["bw_kg"] * ureg.kilogram
d1["bw_lb"] = (d1["bw_kg"].values * ureg.kilogram).to(ureg.pound).magnitude
d1['bw_lb'] = d1['bw_lb'].round(1)

#keeping only significant columns

fulld1 = d1[['name', 'date', 'datetime', 'bw_lb', 'jump_height_cm', 'force_at_zero_vel']]

fulld1 = fulld1.sort_values(by=['name', 'datetime'], ascending=[True, True]).reset_index(drop=True)

fulld1['set'] = fulld1.groupby(['name', 'date']).cumcount() + 1



fulld1.loc[fulld1['set'] == 1, 'days_since_last_test'] = (
    fulld1[fulld1['set'] == 1]
    .groupby('name')['date']
    .diff()
    .dt.days
    .values
)

fulld1['days_since_last_test'] = fulld1['days_since_last_test'].fillna(
    fulld1['set'].apply(lambda x: 0 if x != 1 else -1)
)

fulld1['days_since_last_test'] = fulld1['days_since_last_test'].fillna(0)


#d1 = d1.sort_values(by='datetime', ascending=True)

fulld1['bw_change'] = fulld1.groupby('name')['bw_lb'].diff()


fulld1['jump_height_change'] = (
    fulld1.groupby('name')['jump_height_cm'].diff()
)

fulld1['jump_height_change_per_day'] = (
    fulld1['jump_height_change'] /
    fulld1['days_since_last_test']
)

fulld1['jump_height_change_per_day'] = (
    fulld1['jump_height_change_per_day']
    .replace([np.inf, -np.inf], np.nan)
)

fulld1['bw_change_per_day'] = fulld1['bw_change'] / fulld1['days_since_last_test']

fulld1['bw_change_per_day'] = fulld1['bw_change_per_day'].replace([np.inf, -np.inf], np.nan)

fulld1['is_anomaly'] = 0


# names = ['mcallister burke', 'ian scott', 'aidan plummer', 'emma olausson', 'tyson grimm']


# drop_cols = ['name', 'date', 'datetime',]

# fulld1 = fulld1.drop(columns=drop_cols)

# for col in fulld1:
#     inf_count = np.isinf(fulld1[col]).sum()
#     if inf_count > 0:
#         print(f"{col}: {inf_count} infinity values")
# else:
#     print("0")

# anom = fulld1[
#     fulld1['is_anomaly'] == 1
# ]
fulld1

,name,date,datetime,bw_lb,jump_height_cm,force_at_zero_vel,set,days_since_last_test,bw_change,jump_height_change,jump_height_change_per_day,bw_change_per_day,is_anomaly
0,aaron power,2024-01-23,2024-01-23 08:49:00,162.1,41.969734,2016.445474,1,-1.0,NaN,NaN,NaN,NaN,0
1,abayomi babalola,2024-04-08,2024-04-08 14:25:00,219.1,40.531239,2665.327931,1,-1.0,NaN,NaN,NaN,NaN,0
2,abayomi babalola,2024-04-10,2024-04-10 08:46:00,219.4,42.815915,2757.147258,1,2.0,0.3,2.284676,1.142338,0.150000,0
3,abayomi babalola,2024-04-15,2024-04-15 14:22:00,219.2,41.563591,2704.433521,1,5.0,-0.2,-1.252324,-0.250465,-0.040000,0
4,abayomi babalola,2024-04-17,2024-04-17 08:40:00,223.5,40.162056,2682.577615,1,2.0,4.3,-1.401535,-0.700767,2.150000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
23400,zyion brown,2025-10-29,2025-10-29 08:20:00,246.7,42.717731,3349.450791,1,3.0,-5.1,9.385441,3.128480,-1.700000,0
23401,zyion brown,2025-11-02,2025-11-02 12:16:00,251.7,38.390091,3065.137218,1,4.0,5.0,-4.327640,-1.081910,1.250000,0
23402,zyion brown,2025-11-05,2025-11-05 08:14:00,248.2,43.188406,3282.850191,1,3.0,-3.5,4.798315,1.599438,-1.166667,0
23403,zyion brown,2025-11-16,2025-11-16 12:31:00,254.2,38.215138,3148.763451,1,11.0,6.0,-4.973268,-0.452115,0.545455,0


In [50]:
#invalid tests non olympic

d3 = d3.dropna(subset=['date'])

d3['date'] = pd.to_datetime(d3['date'], errors='coerce')
d3["Unnamed: 3"].unique()


d3['set'] = np.select(
    [
        d3['Unnamed: 3'].str.contains('1st', case=False, na=False),
        d3['Unnamed: 3'].str.contains('2nd', case=False, na=False),
    ],
    [
        1,
        2,
        
    ],
    default= 1
)

d3['is_anomaly'] = 1


#non_olympic = d3[~d3['name'].isin(d4['name'])]


d3 = d3[['name', 'date', 'set', 'is_anomaly']].reset_index(drop=True)


print(d3['is_anomaly'].sum())
d3


131


,name,date,set,is_anomaly
0,fiona o'keefe,2024-01-10,1,1
1,catherine erb,2024-11-01,1,1
2,caroline haggerty,2026-01-05,2,1
3,kate riley,2026-02-23,2,1
4,riley imamura,2025-10-14,2,1
...,...,...,...,...
126,cameron davenport,2025-07-15,1,1
127,vasean washington,2024-05-08,1,1
128,vasean washington,2025-07-01,1,1
129,khari mayeux,2025-07-15,1,1


In [51]:
# search_names = ['aidan plummer', 'emma olausson', 'tyson grimm', 'ian scott', 'mcallister burke']

# for name in search_names:
#     print(f"\n{'='*40}")
#     print(f"  {name}")
#     print(f"{'='*40}")
    
#     # What the label file says
#     label_rows = d3[d3['name'] == name][['name', 'date', 'set']]
#     print(f"  Label file says:")
#     print(label_rows.to_string())
    
#     # What exists in the full dataset
#     dataset_rows = fulld1[fulld1['name'] == name][['name', 'date', 'set']]
#     print(f"\n  Dataset has:")
#     print(dataset_rows.to_string())

In [52]:
#invalid tests olympic

d4 = d4.dropna(subset=['date'])

d4['date'] = pd.to_datetime(d4['date'], errors='coerce')
d4["Unnamed: 4"].unique()


d4['set'] = np.select(
    [
        d4['Unnamed: 4'].str.contains('1st', case=False, na=False),
        d4['Unnamed: 4'].str.contains('2nd', case=False, na=False),
    ],
    [
        1,
        2,
        
    ],
    default= 1
)

d4['is_anomaly'] = 1


olympic = d4[['sport', 'name', 'date', 'set', 'is_anomaly']].reset_index(drop=True)

olympic

,sport,name,date,set,is_anomaly
0,WLAX,fiona o'keefe,2024-01-10,1,1
1,WLAX,catherine erb,2024-11-01,1,1
2,WLAX,caroline haggerty,2026-01-05,2,1
3,WLAX,kate riley,2026-02-23,2,1
4,VB,riley imamura,2025-10-14,2,1
...,...,...,...,...,...
71,MLAX,kenneth edwards,2025-05-12,1,1
72,MLAX,john stenberg,2026-01-05,1,1
73,MLAX,quinn stankovits,2025-10-20,1,1
74,MLAX,quinn stankovits,2026-03-09,1,1


In [53]:
check = d3.merge(
    fulld1[['name', 'date', 'set']],
    on=['name', 'date', 'set'],
    how='left',
    indicator=True
)

matched = check[check['_merge'] == 'both']
print(f"Matched: {len(matched)}")

unmatched = check[check['_merge'] == 'left_only']
print(f"Unmatched: {len(unmatched)}")
print("\nUnmatched rows:")
print(unmatched[['name', 'date', 'set']].to_string())

Matched: 126
Unmatched: 5

Unmatched rows:
                 name       date  set
6       emma olausson        NaT    1
29      aidan plummer 2025-10-15    2
30   mcallister burke        NaT    1
81          ian scott        NaT    1
122       tyson grimm 2025-05-01    1


In [54]:
check2 = olympic.merge(
    fulld1[['name', 'date', 'set']],
    on=['name', 'date', 'set'],
    how='left',
    indicator=True
)

matched2 = check2[check2['_merge'] == 'both']
print(f"\nD4 Matched: {len(matched2)}")

unmatched2 = check2[check2['_merge'] == 'left_only']
print(f"D4 Unmatched: {len(unmatched2)}")
print("\nD4 Unmatched rows:")
print(unmatched2[['name', 'date', 'set']].to_string())


D4 Matched: 73
D4 Unmatched: 3

D4 Unmatched rows:
                name       date  set
6      emma olausson        NaT    1
29     aidan plummer 2025-10-15    2
30  mcallister burke        NaT    1


In [55]:
# result = non_olympic[
#     ~non_olympic.set_index(['name', 'set', 'date']).index.isin(
#         fulld1.set_index(['name', 'set', 'date']).index
#     )
# ]

# result
# #fulld1.loc[(fulld1['date'] == '2025-05-01')]



In [56]:
# result = olympic[
#     ~olympic.set_index(['name', 'set', 'date']).index.isin(
#         fulld1.set_index(['name', 'set', 'date']).index
#     )
# ]

# result
# fulld1.loc[d4["name"] == "aidan plummer"]


In [57]:
# test_d = olympic['name']

# train_d = non_olympic ['name']

# test_d

In [58]:
print(2377 + 2734 )

5111


In [59]:
#

In [60]:
# test_train_data = fulld1[
#     fulld1['name'].isin(list(d3['name']))
# ]
# exclude = test_train_data['name']

# modelling_data = fulld1[~fulld1['name'].isin(exclude)]

# #test_train_data['is_anomaly'] = test_train_data['is_anomaly'].fillna(0).astype(int)





# test_train_data = test_train_data.merge(d3, on=['name', 'date', 'set'], how='left')
# #test_train_data = test_train_data.merge(olympic, on=['name', 'date', 'set'], how='left')

# test_train_data['is_anomaly'] = test_train_data[
#     ['is_anomaly_x', 'is_anomaly_y']
# ].max(axis=1).fillna(0).astype(int)

# test_train_data = test_train_data.drop(columns=['is_anomaly_x', 'is_anomaly_y'])


# # test_train_data = test_train_data.drop(columns=['is_anomaly_x', 'is_anomaly_y'])


# test_train_data

# # print(len(good2))

# # print(len(good))



# # for i in good:
# #     if i not in good2:
# #         print(i)
        

# print(test_train_data['is_anomaly'].sum())
# #merged = olympic_keys.merge(df, on=['name', 'date', 'set'], how='left', indicator=True)


# #n = test_train_data.loc[test_train_data['is_anomaly'] == 1, 'name']


# # noe = []
# # for i in n:
# #     if i not in train_d:
# #        noe.append(i)

# #     elif i not in test_d:
# #         noe.append(i)


# # n

In [61]:
# for i in non_olympic['name']:
#     if i not in training_data['name'].unique():
#         print(i)

In [62]:
fulld1['is_anomaly'].sum()

np.int64(0)

In [63]:

# #Fiona okeefe spelt wrong
# #christopher miller instead of chris miller
# d2 = d2.rename(columns={
#     "Unnamed: 0": "row_id",
#     "Name":       "name",
#     "Date":       "date",
#     "ExternalId": "external_id",
#     "BW":         "bw_kg",
#     "JH":         "jump_height_cm",
#     "F0":         "force_at_zero_vel",
#     "Sport":      "sport",
# })


# d2['date'] = pd.to_datetime(d2['date'])

# d2

# d2['sport'].unique()

# d2.loc[d2["name"] == "aidan plummer"]



In [64]:

    
# combined_df = pd.concat([d3, d4])

# combined_df = combined_df[["name", "date", "sport"]]


# # Identify all rows that are duplicates across the entire set
# duplicates = combined_df.drop_duplicates(keep=False)

# duplicates
# #duplicates['name'].unique()

In [65]:
fulld1.to_csv('cleaned_data/fulld1.csv', index=False)
#modelling_data.to_csv('cleaned_data/modelling_data.csv', index=False)
#test_train_data.to_csv('cleaned_data/test_train_data.csv', index=False)
d3.to_csv('cleaned_data/d3.csv', index=False)
olympic.to_csv('cleaned_data/d4.csv', index=False)